# Final Model C — Train, Save, Download and Predict

This notebook is a standalone deployment build for the selected final Model C.

## Difference between the two model files

| Model | Fitting data | Main purpose | Use for new prediction |
|---|---:|---|---|
| Historical train-only model | Original Train: 3,886 | Exact seed-42 reproduction and implementation audit | No |
| Full-development model | Train + Validation: 4,719 | Final post-selection fit and deployment | **Yes** |

Both models use the same:

```text
13-feature Stage-1 Random Forest + ROS
Direct-six Stage-2 Gradient Boosting + OBU
14 unique patient inputs
Thresholds 0.44 / 0.50
Random seed 42
```

The full-development model uses 833 additional validation participants.

More fitting data can improve robustness on future participants, but it does not guarantee a better prediction for every person. On the already inspected historical test:

| Metric | Train-only audit model | Full-development model |
|---|---:|---:|
| Accuracy | 0.723890 | 0.709484 |
| Balanced Accuracy | 0.668733 | 0.660245 |
| Macro-F1 | 0.566696 | 0.556058 |
| Low recall | 0.755072 | 0.739130 |
| Moderate recall | 0.514286 | 0.504762 |
| High recall | 0.736842 | 0.736842 |
| High→Low | 2 | 2 |

The historical test difference should not be treated as a fresh final model-selection experiment because it has already been inspected. For future inference, use:

```text
FINAL_MODEL_C_FULL_DEVELOPMENT.joblib
```

# What this notebook does

```text
1. Locate and hash the exact modelling CSV.
2. Recreate the original Train / Validation / Test split.
3. Train the historical train-only Model C.
4. Require exact historical metric reproduction.
5. Train Model C on Train + Validation.
6. Require exact full-development reference reproduction.
7. Remove evaluation arrays from the deployment package.
8. Save FINAL_MODEL_C_FULL_DEVELOPMENT.joblib.
9. Reload the file and verify exact predictions and probabilities.
10. Run an illustrative single-patient prediction.
11. Create a complete downloadable deployment ZIP.
```

The deployment ZIP contains:

```text
FINAL_MODEL_C_FULL_DEVELOPMENT.joblib
predict_model_c.py
sample_patient_input.csv
MODEL_C_INPUT_SCHEMA.csv
requirements_inference.txt
MODEL_CARD.md
README_BANGLA.txt
deployment_manifest.json
```

# Kaggle run instructions

Attach:

```text
nhanes_2021_2023_adult_kidney_risk_modeling_leakage_aware.csv
```

Settings:

```text
Accelerator: None
Internet: Off
```

Run:

```text
Session → Restart Session → Run All
```

The following messages must appear:

```text
Exact dataset and split audit: PASS
Historical seed-42 Model C reproduction: PASS
Full-development Model C reproduction: PASS
Deployment package reload verification: PASS
Deployment bundle creation: PASS
```

The last cell provides links to the model and complete ZIP.

In [1]:
# ============================================================
# Cell 1 — Reproducible dependencies
# ============================================================
%pip install -q "scikit-learn==1.6.1" "imbalanced-learn==0.14.1" "joblib==1.5.3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================================
# Cell 2 — Imports, profiles and output folders
# ============================================================
import sys
import gc
import os
import json
import time
import math
import shutil
import zipfile
import hashlib
import platform
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import (
    train_test_split,
    RepeatedStratifiedKFold,
)
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
)
from sklearn.neighbors import NearestNeighbors
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    roc_curve,
    precision_recall_curve,
    auc,
)
from imblearn.over_sampling import RandomOverSampler

import sklearn
import imblearn
import joblib

try:
    from IPython.display import display, FileLink
except Exception:
    display = print
    FileLink = None

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

RUN_PROFILE = "FULL"  # fixed deployment build

RUN_SEED42_TEST = True
RUN_FIVE_SEED_TEST = True
RUN_REPEATED_CV = True
RUN_THRESHOLD_ROBUSTNESS = True
RUN_SUBGROUP_AUDIT = True
SAVE_FULL_DEVELOPMENT_MODEL = True

RUN_EXTERNAL_VALIDATION = False
EXTERNAL_CSV_PATH = ""

RESUME_CV_CACHE = True
CLEAN_OUTPUT = True

PROFILE_SETTINGS = {
    "QUICK": {
        "cv_repeats": 1,
        "bootstrap_reps": 300,
        "test_seeds": [42],
        "threshold_grid": list(
            np.round(
                np.arange(0.40, 0.6001, 0.05),
                2,
            )
        ),
        "inference_repeats": 10,
    },
    "FULL": {
        "cv_repeats": 3,
        "bootstrap_reps": 2000,
        "test_seeds": [42, 52, 62, 72, 82],
        "threshold_grid": list(
            np.round(
                np.arange(0.35, 0.6501, 0.01),
                2,
            )
        ),
        "inference_repeats": 30,
    },
    "ROBUST": {
        "cv_repeats": 5,
        "bootstrap_reps": 3000,
        "test_seeds": [42, 52, 62, 72, 82],
        "threshold_grid": list(
            np.round(
                np.arange(0.35, 0.6501, 0.01),
                2,
            )
        ),
        "inference_repeats": 50,
    },
}

if RUN_PROFILE not in PROFILE_SETTINGS:
    raise ValueError(
        f"Unknown RUN_PROFILE: {RUN_PROFILE}"
    )

SETTINGS = PROFILE_SETTINGS[RUN_PROFILE]

SINGLE_SEED = 42
TEST_SEEDS = list(
    SETTINGS["test_seeds"]
)

CV_FOLDS = 5
CV_REPEATS = int(
    SETTINGS["cv_repeats"]
)
CV_RANDOM_STATE = 20260740
CV_MODEL_SEED = 42

BOOTSTRAP_REPS = int(
    SETTINGS["bootstrap_reps"]
)
BOOTSTRAP_RANDOM_STATE = 20260755

LABEL_ORDER = ["low", "moderate", "high"]
TARGET_COLUMN = "kidney_risk"

STAGE1_THRESHOLD = 0.440
STAGE2_THRESHOLD = 0.500

EXPECTED_CSV_SHA256 = (
    "6e697e9ddde62662a66fee0c408f53ecac1459e904181788802bc66424b5c34a"
)

if Path("/kaggle/working").exists():
    BASE_DIR = Path("/kaggle/working")
    ENVIRONMENT = "kaggle"
elif Path("/content").exists():
    BASE_DIR = Path("/content")
    ENVIRONMENT = "colab"
else:
    BASE_DIR = Path.cwd()
    ENVIRONMENT = "local"

OUTPUT_DIR = (
    BASE_DIR
    / "final_model_c_train_save_download"
)
TABLE_DIR = OUTPUT_DIR / "tables"
PLOT_DIR = OUTPUT_DIR / "plots"
MODEL_DIR = OUTPUT_DIR / "models"
PREDICTION_DIR = OUTPUT_DIR / "predictions"
CACHE_DIR = OUTPUT_DIR / "cv_cache"

if CLEAN_OUTPUT and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for folder in [
    OUTPUT_DIR,
    TABLE_DIR,
    PLOT_DIR,
    MODEL_DIR,
    PREDICTION_DIR,
    CACHE_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.width", 460)
pd.set_option("display.max_colwidth", 260)

configuration_payload = {
    "run_profile": RUN_PROFILE,
    "cv_folds": CV_FOLDS,
    "cv_repeats": CV_REPEATS,
    "cv_random_state": CV_RANDOM_STATE,
    "cv_model_seed": CV_MODEL_SEED,
    "bootstrap_reps": BOOTSTRAP_REPS,
    "test_seeds": TEST_SEEDS,
    "stage1_threshold": STAGE1_THRESHOLD,
    "stage2_threshold": STAGE2_THRESHOLD,
    "scikit_learn": sklearn.__version__,
    "imbalanced_learn": imblearn.__version__,
}

CONFIG_FINGERPRINT = hashlib.sha256(
    json.dumps(
        configuration_payload,
        sort_keys=True,
        default=str,
    ).encode("utf-8")
).hexdigest()[:16]

environment = {
    "environment": ENVIRONMENT,
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    **configuration_payload,
    "config_fingerprint": CONFIG_FINGERPRINT,
}

print(json.dumps(environment, indent=2))

(OUTPUT_DIR / "00_ENVIRONMENT.json").write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)


# Deployment notebook controls
RUN_SEED42_TEST = True
RUN_FIVE_SEED_TEST = False
RUN_REPEATED_CV = False
RUN_THRESHOLD_ROBUSTNESS = False
RUN_SUBGROUP_AUDIT = False
SAVE_FULL_DEVELOPMENT_MODEL = True

ROOT_MODEL_PATH = (
    BASE_DIR
    / "FINAL_MODEL_C_FULL_DEVELOPMENT.joblib"
)

DEPLOYMENT_ZIP_PATH = (
    BASE_DIR
    / "FINAL_MODEL_C_DEPLOYMENT_BUNDLE.zip"
)

INFERENCE_FILES_DIR = (
    OUTPUT_DIR
    / "deployment_files"
)

INFERENCE_FILES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "Deployment output folder:",
    OUTPUT_DIR,
)

{
  "environment": "kaggle",
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
  "numpy": "2.0.2",
  "pandas": "2.3.3",
  "run_profile": "FULL",
  "cv_folds": 5,
  "cv_repeats": 3,
  "cv_random_state": 20260740,
  "cv_model_seed": 42,
  "bootstrap_reps": 2000,
  "test_seeds": [
    42,
    52,
    62,
    72,
    82
  ],
  "stage1_threshold": 0.44,
  "stage2_threshold": 0.5,
  "scikit_learn": "1.6.1",
  "imbalanced_learn": "0.14.1",
  "config_fingerprint": "e8d131d6e4eb8f78"
}
Deployment output folder: /kaggle/working/final_model_c_train_save_download


In [3]:
# ============================================================
# Cell 3 — Exact final features, preprocessing groups and parameters
# ============================================================
BASELINE_22 = [
    "age",
    "sex_female",
    "income_poverty_ratio",
    "bmi",
    "waist",
    "mean_sbp",
    "mean_dbp",
    "mean_pulse",
    "glucose",
    "hba1c",
    "blood_urea_nitrogen",
    "uric_acid",
    "serum_albumin",
    "sodium",
    "potassium",
    "hemoglobin",
    "diabetes_yes",
    "borderline_diabetes",
    "hypertension_doctor_told",
    "high_cholesterol_told",
    "taking_bp_medicine",
    "taking_cholesterol_medicine",
]

MODEL_C_STAGE1 = [
    "age",
    "blood_urea_nitrogen",
    "hba1c",
    "taking_bp_medicine",
    "mean_sbp",
    "glucose",
    "hypertension_doctor_told",
    "uric_acid",
    "diabetes_yes",
    "income_poverty_ratio",
    "mean_dbp",
    "hemoglobin",
    "waist",
]

STAGE2_DIRECT6 = [
    "blood_urea_nitrogen",
    "uric_acid",
    "hemoglobin",
    "serum_albumin",
    "mean_sbp",
    "age",
]

BINARY_FEATURES = {
    "sex_female",
    "diabetes_yes",
    "borderline_diabetes",
    "hypertension_doctor_told",
    "high_cholesterol_told",
    "taking_bp_medicine",
    "taking_cholesterol_medicine",
}

RF_PARAMS = {
    "n_estimators": 150,
    "max_depth": 5,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
    "max_samples": 1.0,
}

GBM_PARAMS = {
    "n_estimators": 150,
    "learning_rate": 0.05,
    "max_depth": 2,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "subsample": 1.0,
    "max_features": None,
}

OBU_CURRENT = {
    "k_neighbors": 7,
    "majority_remove_frac": 0.25,
    "target_ratio": 1.0,
}

MODEL_C_NAME = "Final Model C"

MODEL_C_UNIQUE_INPUTS = list(
    dict.fromkeys(
        MODEL_C_STAGE1
        + STAGE2_DIRECT6
    )
)

if len(MODEL_C_STAGE1) != 13:
    raise AssertionError(
        f"Expected 13 Stage-1 features; found "
        f"{len(MODEL_C_STAGE1)}."
    )

if len(STAGE2_DIRECT6) != 6:
    raise AssertionError(
        f"Expected 6 Stage-2 features; found "
        f"{len(STAGE2_DIRECT6)}."
    )

if len(MODEL_C_UNIQUE_INPUTS) != 14:
    raise AssertionError(
        f"Expected 14 unique patient inputs; found "
        f"{len(MODEL_C_UNIQUE_INPUTS)}."
    )

feature_rows = []

for order, feature in enumerate(
    MODEL_C_STAGE1,
    start=1,
):
    feature_rows.append(
        {
            "Feature": feature,
            "Stage 1": True,
            "Stage-1 order": order,
            "Stage 2": (
                feature
                in STAGE2_DIRECT6
            ),
            "Unique model input": True,
        }
    )

for order, feature in enumerate(
    STAGE2_DIRECT6,
    start=1,
):
    if feature not in MODEL_C_STAGE1:
        feature_rows.append(
            {
                "Feature": feature,
                "Stage 1": False,
                "Stage-1 order": np.nan,
                "Stage 2": True,
                "Unique model input": True,
            }
        )

feature_table = (
    pd.DataFrame(feature_rows)
    .sort_values(
        [
            "Stage 1",
            "Stage-1 order",
            "Feature",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

parameter_table = pd.DataFrame(
    [
        {
            "Component": "Stage-1 Random Forest",
            "Parameters": json.dumps(
                RF_PARAMS,
                sort_keys=True,
            ),
        },
        {
            "Component": "Stage-1 RandomOverSampler",
            "Parameters": json.dumps(
                {
                    "sampling_strategy": 1.0,
                    "random_state": "evaluation seed",
                },
                sort_keys=True,
            ),
        },
        {
            "Component": "Stage-2 current OBU",
            "Parameters": json.dumps(
                {
                    **OBU_CURRENT,
                    "random_state": "evaluation seed",
                },
                sort_keys=True,
            ),
        },
        {
            "Component": "Stage-2 Gradient Boosting",
            "Parameters": json.dumps(
                GBM_PARAMS,
                sort_keys=True,
            ),
        },
        {
            "Component": "Cascade thresholds",
            "Parameters": json.dumps(
                {
                    "stage1": STAGE1_THRESHOLD,
                    "stage2": STAGE2_THRESHOLD,
                },
                sort_keys=True,
            ),
        },
    ]
)

display(feature_table)
display(parameter_table)

feature_table.to_csv(
    TABLE_DIR
    / "01_FINAL_FEATURE_TABLE.csv",
    index=False,
)

parameter_table.to_csv(
    TABLE_DIR
    / "02_FINAL_PARAMETER_TABLE.csv",
    index=False,
)

,Feature,Stage 1,Stage-1 order,Stage 2,Unique model input
0,age,True,1.0,True,True
1,blood_urea_nitrogen,True,2.0,True,True
2,hba1c,True,3.0,False,True
3,taking_bp_medicine,True,4.0,False,True
4,mean_sbp,True,5.0,True,True
5,glucose,True,6.0,False,True
6,hypertension_doctor_told,True,7.0,False,True
7,uric_acid,True,8.0,True,True
8,diabetes_yes,True,9.0,False,True
9,income_poverty_ratio,True,10.0,False,True


,Component,Parameters
0,Stage-1 Random Forest,"{""bootstrap"": true, ""max_depth"": 5, ""max_features"": ""sqrt"", ""max_samples"": 1.0, ""min_samples_leaf"": 2, ""min_samples_split"": 2, ""n_estimators"": 150}"
1,Stage-1 RandomOverSampler,"{""random_state"": ""evaluation seed"", ""sampling_strategy"": 1.0}"
2,Stage-2 current OBU,"{""k_neighbors"": 7, ""majority_remove_frac"": 0.25, ""random_state"": ""evaluation seed"", ""target_ratio"": 1.0}"
3,Stage-2 Gradient Boosting,"{""learning_rate"": 0.05, ""max_depth"": 2, ""max_features"": null, ""min_samples_leaf"": 1, ""min_samples_split"": 2, ""n_estimators"": 150, ""subsample"": 1.0}"
4,Cascade thresholds,"{""stage1"": 0.44, ""stage2"": 0.5}"


In [4]:
# ============================================================
# Cell 4 — Load exact dataset and recreate original split
# ============================================================
EXACT_FILENAME = (
    "nhanes_2021_2023_adult_kidney_risk_modeling_leakage_aware.csv"
)

SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path("/content"),
    Path("/mnt/data"),
    Path.cwd(),
]

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()

def discover_exact_csv():
    matches = []

    for root in SEARCH_ROOTS:
        if not root.exists():
            continue

        try:
            matches.extend(
                root.rglob(EXACT_FILENAME)
            )
        except (PermissionError, OSError):
            continue

    matches = sorted(
        {
            path.resolve()
            for path in matches
            if path.is_file()
            and OUTPUT_DIR.resolve()
            not in path.resolve().parents
        },
        key=lambda path: (
            len(str(path)),
            str(path),
        ),
    )

    if not matches:
        raise FileNotFoundError(
            f"{EXACT_FILENAME} was not found."
        )

    hashes = {
        str(path): sha256_file(path)
        for path in matches
    }

    print("Exact-name CSV candidates:")

    for path, value in hashes.items():
        print(" ", path, value)

    if len(set(hashes.values())) > 1:
        raise RuntimeError(
            "Content-different exact-name CSV files were found."
        )

    return matches[0], hashes[str(matches[0])]

DATA_PATH, DATA_SHA256 = discover_exact_csv()

if DATA_SHA256 != EXPECTED_CSV_SHA256:
    raise RuntimeError(
        "CSV SHA256 mismatch.\n"
        f"Observed: {DATA_SHA256}\n"
        f"Expected: {EXPECTED_CSV_SHA256}"
    )

raw = pd.read_csv(DATA_PATH)

if len(raw) != 5552:
    raise AssertionError(
        f"Expected 5,552 source rows; found {len(raw)}."
    )

raw = (
    raw.dropna(subset=[TARGET_COLUMN])
    .reset_index(drop=True)
)

raw[TARGET_COLUMN] = (
    raw[TARGET_COLUMN]
    .astype(str)
    .str.strip()
    .str.lower()
)

sex = pd.to_numeric(
    raw["sex"],
    errors="coerce",
)

raw["sex_female"] = np.where(
    sex.eq(2),
    1.0,
    np.where(
        sex.eq(1),
        0.0,
        np.nan,
    ),
)

diabetes = pd.to_numeric(
    raw["diabetes_doctor_told"],
    errors="coerce",
)

raw["diabetes_yes"] = np.select(
    [
        diabetes.eq(1),
        diabetes.isin([0, 2, 3]),
    ],
    [1.0, 0.0],
    default=np.nan,
)

raw["borderline_diabetes"] = np.select(
    [
        diabetes.eq(3),
        diabetes.isin([0, 1, 2]),
    ],
    [1.0, 0.0],
    default=np.nan,
)

structural_bp_missing = (
    pd.to_numeric(
        raw["hypertension_doctor_told"],
        errors="coerce",
    ).eq(0)
    & raw["taking_bp_medicine"].isna()
)

raw.loc[
    structural_bp_missing,
    "taking_bp_medicine",
] = 0.0

for feature in BASELINE_22:
    if feature not in raw.columns:
        raise KeyError(
            f"Required feature missing: {feature}"
        )

    raw[feature] = pd.to_numeric(
        raw[feature],
        errors="coerce",
    )

positions = np.arange(len(raw))

train_positions, temporary_positions = train_test_split(
    positions,
    test_size=0.30,
    stratify=raw[TARGET_COLUMN],
    random_state=42,
)

validation_positions, test_positions = train_test_split(
    temporary_positions,
    test_size=0.50,
    stratify=raw.iloc[
        temporary_positions
    ][TARGET_COLUMN],
    random_state=42,
)

train_df = (
    raw.iloc[train_positions]
    .reset_index(drop=True)
)

validation_df = (
    raw.iloc[validation_positions]
    .reset_index(drop=True)
)

test_df = (
    raw.iloc[test_positions]
    .reset_index(drop=True)
)

development_df = pd.concat(
    [train_df, validation_df],
    ignore_index=True,
)

def split_row(name, frame):
    counts = (
        frame[TARGET_COLUMN]
        .value_counts()
        .reindex(LABEL_ORDER)
        .fillna(0)
        .astype(int)
    )

    return {
        "Split": name,
        "N": int(len(frame)),
        "Low": int(counts["low"]),
        "Moderate": int(counts["moderate"]),
        "High": int(counts["high"]),
    }

split_summary = pd.DataFrame(
    [
        split_row("Train", train_df),
        split_row("Validation", validation_df),
        split_row("Development", development_df),
        split_row("Test", test_df),
    ]
)

EXPECTED_SPLITS = {
    "Train": (3886, 3222, 491, 173),
    "Validation": (833, 691, 105, 37),
    "Development": (4719, 3913, 596, 210),
    "Test": (833, 690, 105, 38),
}

for row in split_summary.to_dict(
    orient="records"
):
    observed = (
        row["N"],
        row["Low"],
        row["Moderate"],
        row["High"],
    )

    if observed != EXPECTED_SPLITS[
        row["Split"]
    ]:
        raise AssertionError(
            f"{row['Split']} mismatch: {observed}"
        )

display(split_summary)

split_summary.to_csv(
    TABLE_DIR / "03_SPLIT_SUMMARY.csv",
    index=False,
)

dataset_audit = {
    "dataset_path": str(DATA_PATH),
    "sha256": DATA_SHA256,
    "structural_bp_missing_fixed": int(
        structural_bp_missing.sum()
    ),
    "split_summary": split_summary.to_dict(
        orient="records"
    ),
}

(OUTPUT_DIR / "04_DATASET_AUDIT.json").write_text(
    json.dumps(dataset_audit, indent=2),
    encoding="utf-8",
)

print("Exact dataset and split audit: PASS")

Exact-name CSV candidates:
  /kaggle/input/datasets/muhammadsakibsami/nhanes-21-23/nhanes_2021_2023_adult_kidney_risk_modeling_leakage_aware.csv 6e697e9ddde62662a66fee0c408f53ecac1459e904181788802bc66424b5c34a


,Split,N,Low,Moderate,High
0,Train,3886,3222,491,173
1,Validation,833,691,105,37
2,Development,4719,3913,596,210
3,Test,833,690,105,38


Exact dataset and split audit: PASS


In [5]:
# ============================================================
# Cell 5 — Preprocessing, OBU and base-model fitting
# ============================================================
def unique_ordered(values):
    return list(dict.fromkeys(values))

def make_preprocessor(features):
    features = unique_ordered(features)

    binary = [
        feature
        for feature in features
        if feature in BINARY_FEATURES
    ]

    numeric = [
        feature
        for feature in features
        if feature not in BINARY_FEATURES
    ]

    transformers = []

    if numeric:
        transformers.append(
            (
                "numeric",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="median"
                            ),
                        ),
                        (
                            "scaler",
                            StandardScaler(),
                        ),
                    ]
                ),
                numeric,
            )
        )

    if binary:
        transformers.append(
            (
                "binary",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="most_frequent"
                            ),
                        )
                    ]
                ),
                binary,
            )
        )

    return ColumnTransformer(
        transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )

class OBUResampler:
    def __init__(
        self,
        k_neighbors=7,
        majority_remove_frac=0.25,
        target_ratio=1.0,
        random_state=42,
    ):
        self.k_neighbors = int(k_neighbors)
        self.majority_remove_frac = float(
            majority_remove_frac
        )
        self.target_ratio = float(target_ratio)
        self.random_state = int(random_state)

    def fit_resample(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)

        rng = np.random.RandomState(
            self.random_state
        )

        classes, counts = np.unique(
            y,
            return_counts=True,
        )

        if len(classes) != 2:
            return X, y

        minority = classes[
            np.argmin(counts)
        ]
        majority = classes[
            np.argmax(counts)
        ]

        minority_indices = np.where(
            y == minority
        )[0]
        majority_indices = np.where(
            y == majority
        )[0]

        if (
            len(minority_indices) < 3
            or len(majority_indices) < 3
        ):
            return X, y

        neighbour_count = min(
            self.k_neighbors + 1,
            len(y),
        )

        nearest = NearestNeighbors(
            n_neighbors=neighbour_count
        )
        nearest.fit(X)

        neighbours = (
            nearest.kneighbors(
                X[majority_indices],
                return_distance=False,
            )[:, 1:]
        )

        local_minority_fraction = (
            y[neighbours] == minority
        ).mean(axis=1)

        remove_count = int(
            len(majority_indices)
            * self.majority_remove_frac
        )

        kept_majority = (
            majority_indices.copy()
        )

        if remove_count > 0:
            removed = majority_indices[
                np.argsort(
                    -local_minority_fraction
                )[:remove_count]
            ]

            kept_majority = np.setdiff1d(
                majority_indices,
                removed,
                assume_unique=False,
            )

        target_minority_count = int(
            len(kept_majority)
            * self.target_ratio
        )

        if (
            target_minority_count
            <= len(minority_indices)
        ):
            expanded_minority = (
                minority_indices
            )
        else:
            additional = rng.choice(
                minority_indices,
                size=(
                    target_minority_count
                    - len(minority_indices)
                ),
                replace=True,
            )

            expanded_minority = np.concatenate(
                [
                    minority_indices,
                    additional,
                ]
            )

        final_indices = np.concatenate(
            [
                kept_majority,
                expanded_minority,
            ]
        )

        rng.shuffle(final_indices)

        return (
            X[final_indices],
            y[final_indices],
        )

def positive_probability(model, X):
    probabilities = model.predict_proba(X)
    classes = list(model.classes_)

    if 1 not in classes:
        return np.zeros(
            len(X),
            dtype=float,
        )

    return probabilities[
        :,
        classes.index(1),
    ]

def balanced_sample_weights(y):
    y = np.asarray(y)

    classes, counts = np.unique(
        y,
        return_counts=True,
    )

    class_weights = {
        class_value: (
            len(y)
            / (
                len(classes)
                * count
            )
        )
        for class_value, count
        in zip(classes, counts)
    }

    return np.asarray(
        [
            class_weights[value]
            for value in y
        ],
        dtype=float,
    )

def fit_stage1_rf(
    training_frame,
    evaluation_frame,
    features,
    seed,
):
    preprocessor = make_preprocessor(
        features
    )

    X_train = preprocessor.fit_transform(
        training_frame[features]
    )

    X_evaluation = preprocessor.transform(
        evaluation_frame[features]
    )

    y_train = (
        training_frame[TARGET_COLUMN]
        .ne("low")
        .astype(int)
        .to_numpy()
    )

    X_resampled, y_resampled = (
        RandomOverSampler(
            sampling_strategy=1.0,
            random_state=seed,
        ).fit_resample(
            X_train,
            y_train,
        )
    )

    model = RandomForestClassifier(
        **RF_PARAMS,
        random_state=seed,
        n_jobs=-1,
    )

    model.fit(
        X_resampled,
        y_resampled,
    )

    return {
        "features": list(features),
        "preprocessor": preprocessor,
        "model": model,
        "probability": positive_probability(
            model,
            X_evaluation,
        ),
    }

def fit_stage2_baseline_gbm(
    training_frame,
    evaluation_frame,
    seed,
):
    at_risk = (
        training_frame[TARGET_COLUMN]
        .ne("low")
        .to_numpy()
    )

    preprocessor = make_preprocessor(
        STAGE2_DIRECT6
    )

    X_train = preprocessor.fit_transform(
        training_frame.loc[
            at_risk,
            STAGE2_DIRECT6,
        ]
    )

    X_evaluation = preprocessor.transform(
        evaluation_frame[
            STAGE2_DIRECT6
        ]
    )

    y_train = (
        training_frame.loc[
            at_risk,
            TARGET_COLUMN,
        ]
        .eq("high")
        .astype(int)
        .to_numpy()
    )

    X_resampled, y_resampled = (
        OBUResampler(
            **OBU_CURRENT,
            random_state=seed,
        ).fit_resample(
            X_train,
            y_train,
        )
    )

    model = GradientBoostingClassifier(
        **GBM_PARAMS,
        random_state=seed,
    )

    model.fit(
        X_resampled,
        y_resampled,
    )

    return {
        "features": STAGE2_DIRECT6,
        "preprocessor": preprocessor,
        "model": model,
        "probability": positive_probability(
            model,
            X_evaluation,
        ),
    }

In [6]:
# ============================================================
# Cell 7 — Cascade prediction and complete metrics
# ============================================================
METRIC_COLUMNS = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "low_recall",
    "moderate_recall",
    "high_recall",
    "low_precision",
    "moderate_precision",
    "high_precision",
    "low_f1",
    "moderate_f1",
    "high_f1",
    "low_correct",
    "moderate_correct",
    "high_correct",
    "high_to_low",
    "high_to_low_rate",
    "low_to_high",
    "mcc",
    "cohen_kappa",
    "qwk",
    "ordinal_mae",
    "roc_auc_macro",
    "pr_auc_macro",
    "log_loss",
    "brier",
]

def cascade_prediction(
    p_stage1,
    p_stage2,
):
    p_stage1 = np.asarray(
        p_stage1,
        dtype=float,
    )

    p_stage2 = np.asarray(
        p_stage2,
        dtype=float,
    )

    at_risk = (
        p_stage1
        >= STAGE1_THRESHOLD
    )

    prediction = np.full(
        len(p_stage1),
        "low",
        dtype=object,
    )

    prediction[at_risk] = np.where(
        p_stage2[at_risk]
        >= STAGE2_THRESHOLD,
        "high",
        "moderate",
    )

    return prediction

def compose_probabilities(
    p_stage1,
    p_stage2,
):
    p_stage1 = np.clip(
        np.asarray(
            p_stage1,
            dtype=float,
        ),
        0.0,
        1.0,
    )

    p_stage2 = np.clip(
        np.asarray(
            p_stage2,
            dtype=float,
        ),
        0.0,
        1.0,
    )

    probabilities = np.column_stack(
        [
            1.0 - p_stage1,
            p_stage1
            * (
                1.0 - p_stage2
            ),
            p_stage1
            * p_stage2,
        ]
    )

    row_sum = probabilities.sum(
        axis=1,
        keepdims=True,
    )

    return np.divide(
        probabilities,
        row_sum,
        out=np.full_like(
            probabilities,
            1.0 / 3.0,
        ),
        where=row_sum != 0,
    )

def calculate_metrics(
    y_true,
    prediction,
    probability_matrix,
):
    y_true = np.asarray(
        y_true,
        dtype=object,
    )

    prediction = np.asarray(
        prediction,
        dtype=object,
    )

    probability_matrix = np.asarray(
        probability_matrix,
        dtype=float,
    )

    matrix = confusion_matrix(
        y_true,
        prediction,
        labels=LABEL_ORDER,
    )

    recalls = recall_score(
        y_true,
        prediction,
        labels=LABEL_ORDER,
        average=None,
        zero_division=0,
    )

    precisions = precision_score(
        y_true,
        prediction,
        labels=LABEL_ORDER,
        average=None,
        zero_division=0,
    )

    class_f1 = f1_score(
        y_true,
        prediction,
        labels=LABEL_ORDER,
        average=None,
        zero_division=0,
    )

    true_ordinal = np.asarray(
        [
            LABEL_ORDER.index(label)
            for label in y_true
        ]
    )

    predicted_ordinal = np.asarray(
        [
            LABEL_ORDER.index(label)
            for label in prediction
        ]
    )

    y_binary = label_binarize(
        y_true,
        classes=LABEL_ORDER,
    )

    high_support = int(
        matrix[2, :].sum()
    )

    metrics = {
        "n": int(len(y_true)),
        "accuracy": float(
            accuracy_score(
                y_true,
                prediction,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                prediction,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                prediction,
                labels=LABEL_ORDER,
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_true,
                prediction,
                labels=LABEL_ORDER,
                average="weighted",
                zero_division=0,
            )
        ),
        "low_recall": float(
            recalls[0]
        ),
        "moderate_recall": float(
            recalls[1]
        ),
        "high_recall": float(
            recalls[2]
        ),
        "low_precision": float(
            precisions[0]
        ),
        "moderate_precision": float(
            precisions[1]
        ),
        "high_precision": float(
            precisions[2]
        ),
        "low_f1": float(
            class_f1[0]
        ),
        "moderate_f1": float(
            class_f1[1]
        ),
        "high_f1": float(
            class_f1[2]
        ),
        "low_correct": int(
            matrix[0, 0]
        ),
        "moderate_correct": int(
            matrix[1, 1]
        ),
        "high_correct": int(
            matrix[2, 2]
        ),
        "high_to_low": int(
            matrix[2, 0]
        ),
        "high_to_low_rate": (
            float(
                matrix[2, 0]
                / high_support
            )
            if high_support
            else np.nan
        ),
        "low_to_high": int(
            matrix[0, 2]
        ),
        "mcc": float(
            matthews_corrcoef(
                y_true,
                prediction,
            )
        ),
        "cohen_kappa": float(
            cohen_kappa_score(
                y_true,
                prediction,
            )
        ),
        "qwk": float(
            cohen_kappa_score(
                true_ordinal,
                predicted_ordinal,
                weights="quadratic",
            )
        ),
        "ordinal_mae": float(
            np.mean(
                np.abs(
                    true_ordinal
                    - predicted_ordinal
                )
            )
        ),
        "roc_auc_macro": float(
            roc_auc_score(
                y_binary,
                probability_matrix,
                average="macro",
                multi_class="ovr",
            )
        ),
        "pr_auc_macro": float(
            average_precision_score(
                y_binary,
                probability_matrix,
                average="macro",
            )
        ),
        "log_loss": float(
            -np.mean(
                np.log(
                    np.clip(
                        probability_matrix[
                            np.arange(
                                len(y_true)
                            ),
                            true_ordinal,
                        ],
                        1e-15,
                        1.0,
                    )
                )
            )
        ),
        "brier": float(
            np.mean(
                np.sum(
                    (
                        probability_matrix
                        - y_binary.astype(float)
                    ) ** 2,
                    axis=1,
                )
            )
        ),
    }

    return metrics, matrix

def evaluate_probabilities(
    y_true,
    p_stage1,
    p_stage2,
):
    prediction = cascade_prediction(
        p_stage1,
        p_stage2,
    )

    probability_matrix = (
        compose_probabilities(
            p_stage1,
            p_stage2,
        )
    )

    metrics, matrix = calculate_metrics(
        y_true,
        prediction,
        probability_matrix,
    )

    return {
        "prediction": prediction,
        "probability_matrix": (
            probability_matrix
        ),
        "metrics": metrics,
        "confusion_matrix": matrix,
        "p_stage1": np.asarray(
            p_stage1,
            dtype=float,
        ),
        "p_stage2": np.asarray(
            p_stage2,
            dtype=float,
        ),
    }

def exact_mcnemar(
    y_true,
    baseline_prediction,
    candidate_prediction,
):
    y_true = np.asarray(
        y_true,
        dtype=object,
    )

    baseline_prediction = np.asarray(
        baseline_prediction,
        dtype=object,
    )

    candidate_prediction = np.asarray(
        candidate_prediction,
        dtype=object,
    )

    baseline_correct = (
        baseline_prediction
        == y_true
    )

    candidate_correct = (
        candidate_prediction
        == y_true
    )

    baseline_only = int(
        np.sum(
            baseline_correct
            & ~candidate_correct
        )
    )

    candidate_only = int(
        np.sum(
            ~baseline_correct
            & candidate_correct
        )
    )

    discordant = (
        baseline_only
        + candidate_only
    )

    p_value = (
        float(
            binomtest(
                min(
                    baseline_only,
                    candidate_only,
                ),
                n=discordant,
                p=0.5,
                alternative="two-sided",
            ).pvalue
        )
        if discordant
        else 1.0
    )

    return {
        "Baseline-only correct": (
            baseline_only
        ),
        "Candidate-only correct": (
            candidate_only
        ),
        "Discordant": discordant,
        "Exact McNemar p-value": (
            p_value
        ),
    }

print("Evaluation functions ready.")

Evaluation functions ready.


In [7]:
# ============================================================
# Cell 7 — Historical train-only reproduction audit
# ============================================================
historical_stage1 = fit_stage1_rf(
    train_df,
    test_df,
    MODEL_C_STAGE1,
    seed=SINGLE_SEED,
)

historical_stage2 = (
    fit_stage2_baseline_gbm(
        train_df,
        test_df,
        seed=SINGLE_SEED,
    )
)

historical_evaluation = (
    evaluate_probabilities(
        test_df[
            TARGET_COLUMN
        ],
        historical_stage1[
            "probability"
        ],
        historical_stage2[
            "probability"
        ],
    )
)

EXPECTED_HISTORICAL = {
    "accuracy": 0.723890,
    "balanced_accuracy": 0.668733,
    "macro_f1": 0.566696,
    "low_recall": 0.755072,
    "moderate_recall": 0.514286,
    "high_recall": 0.736842,
    "low_correct": 521,
    "moderate_correct": 54,
    "high_correct": 28,
    "high_to_low": 2,
}

integer_metrics = {
    "low_correct",
    "moderate_correct",
    "high_correct",
    "high_to_low",
}

historical_audit_rows = []
historical_failures = []

observed_metrics = (
    historical_evaluation[
        "metrics"
    ]
)

for metric, expected in (
    EXPECTED_HISTORICAL.items()
):
    observed = (
        observed_metrics[
            metric
        ]
    )

    if metric in integer_metrics:
        passed = (
            int(observed)
            == int(expected)
        )
    else:
        passed = (
            round(
                float(observed),
                6,
            )
            == round(
                float(expected),
                6,
            )
        )

    historical_audit_rows.append(
        {
            "Metric": metric,
            "Observed": observed,
            "Expected": expected,
            "Pass": passed,
        }
    )

    if not passed:
        historical_failures.append(
            {
                "metric": metric,
                "observed": float(
                    observed
                ),
                "expected": float(
                    expected
                ),
            }
        )

historical_audit = pd.DataFrame(
    historical_audit_rows
)

display(
    pd.DataFrame(
        [
            observed_metrics
        ]
    ).round(6)
)

display(
    historical_audit
)

historical_audit.to_csv(
    TABLE_DIR
    / "03_HISTORICAL_REPRODUCTION_AUDIT.csv",
    index=False,
)

(
    OUTPUT_DIR
    / "04_HISTORICAL_FAILURES.json"
).write_text(
    json.dumps(
        historical_failures,
        indent=2,
    ),
    encoding="utf-8",
)

if historical_failures:
    raise RuntimeError(
        "Historical Model C reproduction failed. "
        "The deployment model will not be saved."
    )

print(
    "Historical seed-42 "
    "Model C reproduction: PASS"
)

,n,accuracy,balanced_accuracy,macro_f1,weighted_f1,low_recall,moderate_recall,high_recall,low_precision,moderate_precision,high_precision,low_f1,moderate_f1,high_f1,low_correct,moderate_correct,high_correct,high_to_low,high_to_low_rate,low_to_high,mcc,cohen_kappa,qwk,ordinal_mae,roc_auc_macro,pr_auc_macro,log_loss,brier
0,833,0.72389,0.668733,0.566696,0.760836,0.755072,0.514286,0.736842,0.940433,0.258373,0.4,0.837621,0.343949,0.518519,521,54,28,2,0.052632,22,0.362057,0.332495,0.491868,0.304922,0.792867,0.586494,0.580178,0.322225


,Metric,Observed,Expected,Pass
0,accuracy,0.723890,0.723890,True
1,balanced_accuracy,0.668733,0.668733,True
2,macro_f1,0.566696,0.566696,True
3,low_recall,0.755072,0.755072,True
4,moderate_recall,0.514286,0.514286,True
5,high_recall,0.736842,0.736842,True
6,low_correct,521.000000,521.000000,True
7,moderate_correct,54.000000,54.000000,True
8,high_correct,28.000000,28.000000,True
9,high_to_low,2.000000,2.000000,True


Historical seed-42 Model C reproduction: PASS


In [8]:
# ============================================================
# Cell 8 — Train and save the full-development model
# ============================================================
deployment_stage1 = fit_stage1_rf(
    development_df,
    test_df,
    MODEL_C_STAGE1,
    seed=SINGLE_SEED,
)

deployment_stage2 = (
    fit_stage2_baseline_gbm(
        development_df,
        test_df,
        seed=SINGLE_SEED,
    )
)

deployment_evaluation = (
    evaluate_probabilities(
        test_df[
            TARGET_COLUMN
        ],
        deployment_stage1[
            "probability"
        ],
        deployment_stage2[
            "probability"
        ],
    )
)

EXPECTED_FULL_DEVELOPMENT = {
    "accuracy": 0.709484,
    "balanced_accuracy": 0.660245,
    "macro_f1": 0.556058,
    "low_recall": 0.739130,
    "moderate_recall": 0.504762,
    "high_recall": 0.736842,
    "high_to_low": 2,
}

deployment_audit_rows = []
deployment_failures = []

deployment_metrics = (
    deployment_evaluation[
        "metrics"
    ]
)

for metric, expected in (
    EXPECTED_FULL_DEVELOPMENT.items()
):
    observed = (
        deployment_metrics[
            metric
        ]
    )

    if metric == "high_to_low":
        passed = (
            int(observed)
            == int(expected)
        )
    else:
        passed = (
            round(
                float(observed),
                6,
            )
            == round(
                float(expected),
                6,
            )
        )

    deployment_audit_rows.append(
        {
            "Metric": metric,
            "Observed": observed,
            "Expected": expected,
            "Pass": passed,
        }
    )

    if not passed:
        deployment_failures.append(
            {
                "metric": metric,
                "observed": float(
                    observed
                ),
                "expected": float(
                    expected
                ),
            }
        )

deployment_audit = pd.DataFrame(
    deployment_audit_rows
)

comparison_table = pd.DataFrame(
    [
        {
            "Package": (
                "Historical Train-only audit"
            ),
            "Training N": len(
                train_df
            ),
            **{
                metric: historical_evaluation[
                    "metrics"
                ][
                    metric
                ]
                for metric in [
                    "accuracy",
                    "balanced_accuracy",
                    "macro_f1",
                    "low_recall",
                    "moderate_recall",
                    "high_recall",
                    "high_to_low",
                ]
            },
        },
        {
            "Package": (
                "Full-development deployment"
            ),
            "Training N": len(
                development_df
            ),
            **{
                metric: deployment_metrics[
                    metric
                ]
                for metric in [
                    "accuracy",
                    "balanced_accuracy",
                    "macro_f1",
                    "low_recall",
                    "moderate_recall",
                    "high_recall",
                    "high_to_low",
                ]
            },
        },
    ]
)

display(
    pd.DataFrame(
        [
            deployment_metrics
        ]
    ).round(6)
)

display(
    deployment_audit
)

display(
    comparison_table.round(6)
)

deployment_audit.to_csv(
    TABLE_DIR
    / "05_FULL_DEVELOPMENT_REPRODUCTION_AUDIT.csv",
    index=False,
)

comparison_table.to_csv(
    TABLE_DIR
    / "06_TRAIN_ONLY_VS_DEPLOYMENT.csv",
    index=False,
)

(
    OUTPUT_DIR
    / "07_FULL_DEVELOPMENT_FAILURES.json"
).write_text(
    json.dumps(
        deployment_failures,
        indent=2,
    ),
    encoding="utf-8",
)

if deployment_failures:
    raise RuntimeError(
        "Full-development reference "
        "metrics did not reproduce. "
        "The deployment model will not be saved."
    )

# Remove evaluation-specific probability arrays from the package.
clean_stage1 = {
    "features": list(
        deployment_stage1[
            "features"
        ]
    ),
    "preprocessor": (
        deployment_stage1[
            "preprocessor"
        ]
    ),
    "model": (
        deployment_stage1[
            "model"
        ]
    ),
}

clean_stage2 = {
    "features": list(
        deployment_stage2[
            "features"
        ]
    ),
    "preprocessor": (
        deployment_stage2[
            "preprocessor"
        ]
    ),
    "model": (
        deployment_stage2[
            "model"
        ]
    ),
}

deployment_package = {
    "package_version": "1.0",
    "model_name": (
        "Final Model C"
    ),
    "model_status": (
        "Selected deployment model"
    ),
    "training_scope": (
        "NHANES 2021–2023 "
        "original Train + Validation"
    ),
    "training_n": int(
        len(
            development_df
        )
    ),
    "training_class_counts": {
        str(key): int(value)
        for key, value
        in (
            development_df[
                TARGET_COLUMN
            ]
            .value_counts()
            .reindex(
                LABEL_ORDER
            )
            .fillna(0)
            .astype(int)
            .to_dict()
            .items()
        )
    },
    "seed": int(
        SINGLE_SEED
    ),
    "stage1": clean_stage1,
    "stage2": clean_stage2,
    "stage1_features": list(
        MODEL_C_STAGE1
    ),
    "stage2_features": list(
        STAGE2_DIRECT6
    ),
    "unique_inputs": list(
        MODEL_C_UNIQUE_INPUTS
    ),
    "thresholds": {
        "stage1": (
            STAGE1_THRESHOLD
        ),
        "stage2": (
            STAGE2_THRESHOLD
        ),
    },
    "rf_parameters": dict(
        RF_PARAMS
    ),
    "gbm_parameters": dict(
        GBM_PARAMS
    ),
    "obu_parameters": dict(
        OBU_CURRENT
    ),
    "versions": {
        "python": sys.version,
        "scikit_learn": (
            sklearn.__version__
        ),
        "imbalanced_learn": (
            imblearn.__version__
        ),
        "joblib": (
            joblib.__version__
        ),
    },
    "scientific_boundary": (
        "Kidney-risk screening proxy; "
        "not a confirmed CKD diagnosis."
    ),
}

FINAL_MODEL_PATH = (
    MODEL_DIR
    / "FINAL_MODEL_C_FULL_DEVELOPMENT.joblib"
)

joblib.dump(
    deployment_package,
    FINAL_MODEL_PATH,
    compress=3,
)

shutil.copy2(
    FINAL_MODEL_PATH,
    ROOT_MODEL_PATH,
)

print(
    "Full-development Model C "
    "reproduction: PASS"
)

print(
    "Deployment model saved:"
)

print(
    " ",
    FINAL_MODEL_PATH,
)

print(
    " ",
    ROOT_MODEL_PATH,
)

,n,accuracy,balanced_accuracy,macro_f1,weighted_f1,low_recall,moderate_recall,high_recall,low_precision,moderate_precision,high_precision,low_f1,moderate_f1,high_f1,low_correct,moderate_correct,high_correct,high_to_low,high_to_low_rate,low_to_high,mcc,cohen_kappa,qwk,ordinal_mae,roc_auc_macro,pr_auc_macro,log_loss,brier
0,833,0.709484,0.660245,0.556058,0.749913,0.73913,0.504762,0.736842,0.939227,0.242009,0.394366,0.827251,0.32716,0.513761,510,53,28,2,0.052632,22,0.344927,0.313226,0.479373,0.319328,0.791739,0.576333,0.580072,0.323415


,Metric,Observed,Expected,Pass
0,accuracy,0.709484,0.709484,True
1,balanced_accuracy,0.660245,0.660245,True
2,macro_f1,0.556058,0.556058,True
3,low_recall,0.739130,0.739130,True
4,moderate_recall,0.504762,0.504762,True
5,high_recall,0.736842,0.736842,True
6,high_to_low,2.000000,2.000000,True


,Package,Training N,accuracy,balanced_accuracy,macro_f1,low_recall,moderate_recall,high_recall,high_to_low
0,Historical Train-only audit,3886,0.723890,0.668733,0.566696,0.755072,0.514286,0.736842,2
1,Full-development deployment,4719,0.709484,0.660245,0.556058,0.739130,0.504762,0.736842,2


Full-development Model C reproduction: PASS
Deployment model saved:
  /kaggle/working/final_model_c_train_save_download/models/FINAL_MODEL_C_FULL_DEVELOPMENT.joblib
  /kaggle/working/FINAL_MODEL_C_FULL_DEVELOPMENT.joblib


In [9]:
# ============================================================
# Cell 9 — Reload verification and sample prediction
# ============================================================
def component_probability(
    component,
    frame,
):
    transformed = (
        component[
            "preprocessor"
        ].transform(
            frame[
                component[
                    "features"
                ]
            ]
        )
    )

    return positive_probability(
        component[
            "model"
        ],
        transformed,
    )


def predict_saved_package(
    package,
    frame,
):
    frame = frame.copy()

    missing = [
        feature
        for feature in (
            package[
                "unique_inputs"
            ]
        )
        if feature
        not in frame.columns
    ]

    if missing:
        raise KeyError(
            f"Missing prediction inputs: "
            f"{missing}"
        )

    for feature in (
        package[
            "unique_inputs"
        ]
    ):
        frame[
            feature
        ] = pd.to_numeric(
            frame[
                feature
            ],
            errors="coerce",
        )

    binary_features = {
        "taking_bp_medicine",
        "hypertension_doctor_told",
        "diabetes_yes",
    }

    for feature in binary_features:
        invalid = (
            frame[
                feature
            ].notna()
            & ~frame[
                feature
            ].isin(
                [
                    0,
                    1,
                ]
            )
        )

        if invalid.any():
            raise ValueError(
                f"{feature} must contain "
                "0, 1 or missing."
            )

    p_stage1 = (
        component_probability(
            package[
                "stage1"
            ],
            frame,
        )
    )

    p_stage2 = (
        component_probability(
            package[
                "stage2"
            ],
            frame,
        )
    )

    prediction = cascade_prediction(
        p_stage1,
        p_stage2,
    )

    probabilities = (
        compose_probabilities(
            p_stage1,
            p_stage2,
        )
    )

    return {
        "prediction": prediction,
        "p_stage1": p_stage1,
        "p_stage2": p_stage2,
        "probability_matrix": (
            probabilities
        ),
    }


reloaded_package = joblib.load(
    FINAL_MODEL_PATH
)

reloaded_test_result = (
    predict_saved_package(
        reloaded_package,
        test_df,
    )
)

prediction_match = np.array_equal(
    deployment_evaluation[
        "prediction"
    ],
    reloaded_test_result[
        "prediction"
    ],
)

stage1_max_difference = float(
    np.max(
        np.abs(
            deployment_evaluation[
                "p_stage1"
            ]
            - reloaded_test_result[
                "p_stage1"
            ]
        )
    )
)

stage2_max_difference = float(
    np.max(
        np.abs(
            deployment_evaluation[
                "p_stage2"
            ]
            - reloaded_test_result[
                "p_stage2"
            ]
        )
    )
)

probability_max_difference = float(
    np.max(
        np.abs(
            deployment_evaluation[
                "probability_matrix"
            ]
            - reloaded_test_result[
                "probability_matrix"
            ]
        )
    )
)

reload_audit = {
    "prediction_exact_match": bool(
        prediction_match
    ),
    "stage1_max_absolute_difference": (
        stage1_max_difference
    ),
    "stage2_max_absolute_difference": (
        stage2_max_difference
    ),
    "probability_max_absolute_difference": (
        probability_max_difference
    ),
    "model_sha256": sha256_file(
        FINAL_MODEL_PATH
    ),
    "model_size_bytes": int(
        FINAL_MODEL_PATH.stat().st_size
    ),
}

print(
    json.dumps(
        reload_audit,
        indent=2,
    )
)

(
    OUTPUT_DIR
    / "08_RELOAD_VERIFICATION.json"
).write_text(
    json.dumps(
        reload_audit,
        indent=2,
    ),
    encoding="utf-8",
)

if (
    not prediction_match
    or stage1_max_difference
    > 1e-12
    or stage2_max_difference
    > 1e-12
    or probability_max_difference
    > 1e-12
):
    raise RuntimeError(
        "Saved model reload "
        "verification failed."
    )

print(
    "Deployment package "
    "reload verification: PASS"
)

sample_patient = pd.DataFrame(
    [
        {
            "age": 58,
            "blood_urea_nitrogen": 20.0,
            "hba1c": 6.2,
            "taking_bp_medicine": 1,
            "mean_sbp": 138.0,
            "glucose": 112.0,
            "hypertension_doctor_told": 1,
            "uric_acid": 6.5,
            "diabetes_yes": 0,
            "income_poverty_ratio": 2.1,
            "mean_dbp": 82.0,
            "hemoglobin": 13.8,
            "waist": 97.0,
            "serum_albumin": 4.2,
        }
    ]
)

sample_result = (
    predict_saved_package(
        reloaded_package,
        sample_patient,
    )
)

sample_output = pd.DataFrame(
    {
        "predicted_risk_class": (
            sample_result[
                "prediction"
            ]
        ),
        "stage1_at_risk_score": (
            sample_result[
                "p_stage1"
            ]
        ),
        "stage2_high_given_at_risk_score": (
            sample_result[
                "p_stage2"
            ]
        ),
        "low_score": (
            sample_result[
                "probability_matrix"
            ][
                :,
                0,
            ]
        ),
        "moderate_score": (
            sample_result[
                "probability_matrix"
            ][
                :,
                1,
            ]
        ),
        "high_score": (
            sample_result[
                "probability_matrix"
            ][
                :,
                2,
            ]
        ),
    }
)

display(
    sample_output.round(6)
)

{
  "prediction_exact_match": true,
  "stage1_max_absolute_difference": 2.220446049250313e-16,
  "stage2_max_absolute_difference": 0.0,
  "probability_max_absolute_difference": 2.220446049250313e-16,
  "model_sha256": "5f1363207f75d4eedce08280e8b4a495beada1bb3648dc743e900bfaeb09442a",
  "model_size_bytes": 309650
}
Deployment package reload verification: PASS


,predicted_risk_class,stage1_at_risk_score,stage2_high_given_at_risk_score,low_score,moderate_score,high_score
0,high,0.607881,0.543981,0.392119,0.277205,0.330675


In [10]:
# ============================================================
# Cell 10 — Build the downloadable deployment bundle
# ============================================================
prediction_script = 'from pathlib import Path\nfrom typing import Union\n\nimport joblib\nimport numpy as np\nimport pandas as pd\n\n\nMODEL_PATH = Path(\n    "FINAL_MODEL_C_FULL_DEVELOPMENT.joblib"\n)\n\nBINARY_FEATURES = {\n    "taking_bp_medicine",\n    "hypertension_doctor_told",\n    "diabetes_yes",\n}\n\n\ndef load_model(\n    model_path: Union[str, Path] = MODEL_PATH,\n) -> dict:\n    model_path = Path(model_path)\n\n    if not model_path.exists():\n        raise FileNotFoundError(\n            f"Model file not found: "\n            f"{model_path.resolve()}"\n        )\n\n    package = joblib.load(model_path)\n\n    required_keys = {\n        "stage1",\n        "stage2",\n        "thresholds",\n        "unique_inputs",\n    }\n\n    missing = (\n        required_keys\n        - set(package)\n    )\n\n    if missing:\n        raise KeyError(\n            f"Invalid model package. "\n            f"Missing keys: {sorted(missing)}"\n        )\n\n    return package\n\n\ndef positive_probability(\n    component: dict,\n    frame: pd.DataFrame,\n) -> np.ndarray:\n    transformed = (\n        component["preprocessor"]\n        .transform(\n            frame[\n                component["features"]\n            ]\n        )\n    )\n\n    model = component["model"]\n    classes = list(model.classes_)\n\n    if 1 not in classes:\n        return np.zeros(\n            len(frame),\n            dtype=float,\n        )\n\n    return model.predict_proba(\n        transformed\n    )[:, classes.index(1)]\n\n\ndef validate_input(\n    frame: pd.DataFrame,\n    package: dict,\n) -> pd.DataFrame:\n    if not isinstance(\n        frame,\n        pd.DataFrame,\n    ):\n        raise TypeError(\n            "Input must be a pandas DataFrame."\n        )\n\n    frame = frame.copy()\n    required = package["unique_inputs"]\n\n    missing = [\n        feature\n        for feature in required\n        if feature\n        not in frame.columns\n    ]\n\n    if missing:\n        raise KeyError(\n            f"Missing required columns: "\n            f"{missing}"\n        )\n\n    for feature in required:\n        frame[feature] = pd.to_numeric(\n            frame[feature],\n            errors="coerce",\n        )\n\n    for feature in BINARY_FEATURES:\n        invalid = (\n            frame[feature].notna()\n            & ~frame[feature].isin([0, 1])\n        )\n\n        if invalid.any():\n            values = (\n                frame.loc[\n                    invalid,\n                    feature,\n                ]\n                .unique()\n                .tolist()\n            )\n\n            raise ValueError(\n                f"{feature} accepts only "\n                f"0, 1 or missing. "\n                f"Found: {values}"\n            )\n\n    return frame\n\n\ndef predict_dataframe(\n    input_frame: pd.DataFrame,\n    package: dict,\n) -> pd.DataFrame:\n    frame = validate_input(\n        input_frame,\n        package,\n    )\n\n    p_stage1 = positive_probability(\n        package["stage1"],\n        frame,\n    )\n\n    p_stage2 = positive_probability(\n        package["stage2"],\n        frame,\n    )\n\n    threshold1 = float(\n        package["thresholds"]["stage1"]\n    )\n\n    threshold2 = float(\n        package["thresholds"]["stage2"]\n    )\n\n    at_risk = (\n        p_stage1\n        >= threshold1\n    )\n\n    predicted_class = np.full(\n        len(frame),\n        "low",\n        dtype=object,\n    )\n\n    predicted_class[at_risk] = np.where(\n        p_stage2[at_risk]\n        >= threshold2,\n        "high",\n        "moderate",\n    )\n\n    low_score = (\n        1.0\n        - p_stage1\n    )\n\n    moderate_score = (\n        p_stage1\n        * (\n            1.0\n            - p_stage2\n        )\n    )\n\n    high_score = (\n        p_stage1\n        * p_stage2\n    )\n\n    result = frame.copy()\n\n    result[\n        "predicted_risk_class"\n    ] = predicted_class\n\n    result[\n        "stage1_at_risk_score"\n    ] = p_stage1\n\n    result[\n        "stage2_high_given_at_risk_score"\n    ] = p_stage2\n\n    result["low_score"] = low_score\n    result["moderate_score"] = moderate_score\n    result["high_score"] = high_score\n\n    result[\n        "screening_note"\n    ] = (\n        "Screening-risk proxy; "\n        "not a confirmed CKD diagnosis."\n    )\n\n    return result\n\n\ndef predict_csv(\n    input_csv: Union[str, Path],\n    output_csv: Union[\n        str,\n        Path,\n    ] = "model_c_predictions.csv",\n    model_path: Union[\n        str,\n        Path,\n    ] = MODEL_PATH,\n) -> pd.DataFrame:\n    package = load_model(\n        model_path\n    )\n\n    input_frame = pd.read_csv(\n        input_csv\n    )\n\n    result = predict_dataframe(\n        input_frame,\n        package,\n    )\n\n    result.to_csv(\n        output_csv,\n        index=False,\n    )\n\n    return result\n\n\nif __name__ == "__main__":\n    result = predict_csv(\n        "sample_patient_input.csv",\n        "model_c_predictions.csv",\n    )\n\n    print(\n        result[\n            [\n                "predicted_risk_class",\n                "stage1_at_risk_score",\n                "stage2_high_given_at_risk_score",\n                "low_score",\n                "moderate_score",\n                "high_score",\n            ]\n        ].to_string(\n            index=False\n        )\n    )\n\n    print(\n        "\\nSaved: model_c_predictions.csv"\n    )\n'

INPUT_SCHEMA = [
    {
        "feature": "age",
        "unit_or_encoding": "years",
        "accepted": "numeric",
    },
    {
        "feature": "blood_urea_nitrogen",
        "unit_or_encoding": "mg/dL",
        "accepted": "numeric",
    },
    {
        "feature": "hba1c",
        "unit_or_encoding": "%",
        "accepted": "numeric",
    },
    {
        "feature": "taking_bp_medicine",
        "unit_or_encoding": "No=0, Yes=1",
        "accepted": "0, 1 or missing",
    },
    {
        "feature": "mean_sbp",
        "unit_or_encoding": "mmHg",
        "accepted": "numeric",
    },
    {
        "feature": "glucose",
        "unit_or_encoding": "mg/dL",
        "accepted": "numeric",
    },
    {
        "feature": "hypertension_doctor_told",
        "unit_or_encoding": "No=0, Yes=1",
        "accepted": "0, 1 or missing",
    },
    {
        "feature": "uric_acid",
        "unit_or_encoding": "mg/dL",
        "accepted": "numeric",
    },
    {
        "feature": "diabetes_yes",
        "unit_or_encoding": "No=0, Yes=1",
        "accepted": "0, 1 or missing",
    },
    {
        "feature": "income_poverty_ratio",
        "unit_or_encoding": "NHANES-compatible ratio",
        "accepted": "numeric",
    },
    {
        "feature": "mean_dbp",
        "unit_or_encoding": "mmHg",
        "accepted": "numeric",
    },
    {
        "feature": "hemoglobin",
        "unit_or_encoding": "g/dL",
        "accepted": "numeric",
    },
    {
        "feature": "waist",
        "unit_or_encoding": "cm",
        "accepted": "numeric",
    },
    {
        "feature": "serum_albumin",
        "unit_or_encoding": "g/dL",
        "accepted": "numeric",
    },
]

schema_frame = pd.DataFrame(
    INPUT_SCHEMA
)

schema_frame.to_csv(
    INFERENCE_FILES_DIR
    / "MODEL_C_INPUT_SCHEMA.csv",
    index=False,
)

sample_patient.to_csv(
    INFERENCE_FILES_DIR
    / "sample_patient_input.csv",
    index=False,
)

(
    INFERENCE_FILES_DIR
    / "predict_model_c.py"
).write_text(
    prediction_script,
    encoding="utf-8",
)

requirements_text = (
    "scikit-learn==1.6.1\n"
    "joblib==1.5.3\n"
    "pandas>=2.0\n"
    "numpy>=1.24\n"
)

(
    INFERENCE_FILES_DIR
    / "requirements_inference.txt"
).write_text(
    requirements_text,
    encoding="utf-8",
)

model_card = (
    "# Final Model C — Model Card\n\n"
    "## Artifact\n\n"
    "`FINAL_MODEL_C_FULL_DEVELOPMENT.joblib`\n\n"
    "## Training scope\n\n"
    "NHANES 2021–2023 original Train + Validation.\n\n"
    f"- Training N: {deployment_package['training_n']}\n"
    f"- Low: {deployment_package['training_class_counts']['low']}\n"
    f"- Moderate: {deployment_package['training_class_counts']['moderate']}\n"
    f"- High: {deployment_package['training_class_counts']['high']}\n\n"
    "## Architecture\n\n"
    "- Stage 1: 13-feature Random Forest + RandomOverSampler\n"
    "- Stage 2: direct-six Gradient Boosting + current OBU\n"
    f"- Stage-1 threshold: {STAGE1_THRESHOLD}\n"
    f"- Stage-2 threshold: {STAGE2_THRESHOLD}\n"
    "- Unique inputs: 14\n"
    "- Random seed: 42\n\n"
    "## Intended use\n\n"
    "Low / Moderate / High kidney-risk screening.\n\n"
    "This is a screening-risk proxy, not a confirmed CKD diagnosis.\n\n"
    "The score columns are not calibrated clinical disease percentages.\n"
)

(
    INFERENCE_FILES_DIR
    / "MODEL_CARD.md"
).write_text(
    model_card,
    encoding="utf-8",
)

readme_bangla = (
    "FINAL MODEL C — ব্যবহার নির্দেশনা\n"
    "==================================\n\n"
    "সব file একই folder-এ রাখুন।\n\n"
    "Windows PowerShell:\n"
    "cd D:\\Model_C_Prediction\n"
    "py -3.10 -m venv .venv\n"
    ".venv\\Scripts\\activate\n"
    "pip install -r requirements_inference.txt\n"
    "python predict_model_c.py\n\n"
    "Output: model_c_predictions.csv\n\n"
    "নিজের data ব্যবহার করতে sample_patient_input.csv-এর "
    "একই 14টি column ব্যবহার করুন।\n\n"
    "taking_bp_medicine: No=0, Yes=1\n"
    "hypertension_doctor_told: No=0, Yes=1\n"
    "diabetes_yes: No=0, Yes=1\n\n"
    "Output screening-risk class; confirmed CKD diagnosis নয়।\n"
)

(
    INFERENCE_FILES_DIR
    / "README_BANGLA.txt"
).write_text(
    readme_bangla,
    encoding="utf-8",
)

shutil.copy2(
    FINAL_MODEL_PATH,
    INFERENCE_FILES_DIR
    / FINAL_MODEL_PATH.name,
)

deployment_manifest = {
    "model_file": (
        FINAL_MODEL_PATH.name
    ),
    "model_sha256": sha256_file(
        FINAL_MODEL_PATH
    ),
    "training_scope": (
        deployment_package[
            "training_scope"
        ]
    ),
    "training_n": (
        deployment_package[
            "training_n"
        ]
    ),
    "unique_inputs": (
        deployment_package[
            "unique_inputs"
        ]
    ),
    "thresholds": (
        deployment_package[
            "thresholds"
        ]
    ),
    "files": sorted(
        path.name
        for path in (
            INFERENCE_FILES_DIR.iterdir()
        )
        if path.is_file()
    ),
}

(
    INFERENCE_FILES_DIR
    / "deployment_manifest.json"
).write_text(
    json.dumps(
        deployment_manifest,
        indent=2,
    ),
    encoding="utf-8",
)

if DEPLOYMENT_ZIP_PATH.exists():
    DEPLOYMENT_ZIP_PATH.unlink()

with zipfile.ZipFile(
    DEPLOYMENT_ZIP_PATH,
    "w",
    compression=(
        zipfile.ZIP_DEFLATED
    ),
) as archive:
    for path in sorted(
        INFERENCE_FILES_DIR.iterdir()
    ):
        if path.is_file():
            archive.write(
                path,
                arcname=path.name,
            )

print(
    "Deployment bundle creation: PASS"
)

print(
    "\nDownload the model only:"
)

display(
    FileLink(
        str(
            ROOT_MODEL_PATH
        )
    )
)

print(
    "\nRecommended — download "
    "the complete bundle:"
)

display(
    FileLink(
        str(
            DEPLOYMENT_ZIP_PATH
        )
    )
)

print(
    "\nModel SHA256:",
    deployment_manifest[
        "model_sha256"
    ],
)

print(
    "Bundle:",
    DEPLOYMENT_ZIP_PATH,
)

Deployment bundle creation: PASS

Download the model only:


/kaggle/working/FINAL_MODEL_C_FULL_DEVELOPMENT.joblib


Recommended — download the complete bundle:


/kaggle/working/FINAL_MODEL_C_DEPLOYMENT_BUNDLE.zip


Model SHA256: 5f1363207f75d4eedce08280e8b4a495beada1bb3648dc743e900bfaeb09442a
Bundle: /kaggle/working/FINAL_MODEL_C_DEPLOYMENT_BUNDLE.zip


# Local prediction use

Download and extract:

```text
FINAL_MODEL_C_DEPLOYMENT_BUNDLE.zip
```

Windows PowerShell:

```powershell
cd D:\Model_C_Prediction

py -3.10 -m venv .venv
.venv\Scripts\activate

pip install -r requirements_inference.txt
python predict_model_c.py
```

The sample result is written to:

```text
model_c_predictions.csv
```

To predict another CSV:

```python
from predict_model_c import predict_csv

result = predict_csv(
    "my_patients.csv",
    "my_predictions.csv",
)
```

The input CSV must use the same 14 columns as `sample_patient_input.csv`.

Prediction logic:

```text
Stage-1 score < 0.44
    → Low

Stage-1 score ≥ 0.44 and Stage-2 score < 0.50
    → Moderate

Stage-1 score ≥ 0.44 and Stage-2 score ≥ 0.50
    → High
```

These are screening-model scores, not calibrated clinical disease percentages.